# Combining Multiple Sources of Training Data

We have three types of trainign data: 
1. False Beluga from 216 and 237 via CSV
2. Sub-1000hz beluga calls via CSV
3. Traditional PG Detections via DosCiba DB

The goal of this effort is to develop a routine to combine multiple sources of training data, including the false beluga and sub-1000hz calls. The spectrogram generation routine developed by Microsoft is the consumer of this work. Put simply, instead of reading in one CSV file as a dataframe, we will pull data from these sources and merge them into one large dataframe. 

Ensuring that the times The spectrogram generation routine utilizes the UTC detection time to match the detection to a specific wave file. 

A few things to note. First, the times specified in filenames and thusly, detections are all labeled UTC but are actually local time. As such, we must carefully evaluate the data to determine what localization the listed times are. Specifically, we should anticipate that both false belugas (i.e. data from 216 and 237) and sub-1000hz UTC times are local AK time. In contrast, all times stored n the DosCiba DB are true UTC time.


In [1]:
import sqlite3 as sq3
import pandas as pd


## Get Pamguard data from database

In [2]:
dosdb_path = r"D:\beluga\Data\Database\dosciba_workingdb.db"
dosdb = sq3.connect(dosdb_path)

pg_dataframe = pd.read_sql_query("select * from pamguard where Deployment in (201,206,213,214,215,216,218,223,237)", dosdb, parse_dates=['Original'])
pg_dataframe.head()


,Id,UID,Original,UTCMilliseconds,PCLocalTime,PCTime,ChannelBitmap,SequenceBitmap,UpdateOf,channelMap,...,amplitude,detectionType,bearingAmbiguity,Fixed Localized,Species,Comment,Deployment,Local_Time,UTC_Time,detection_not_on_datamap
0,1,20000001,2017-09-30 15:47:14.053,53,2017-09-30 08:47:14.053,2018-10-24 00:14:32.985,1,,0,1,...,151.483972,,,,F,,201,2017-09-30 15:47:14.053000-07:00,2017-09-30 22:47:14.053000+00:00,None
1,2,20000002,2017-09-30 15:47:18.299,299,2017-09-30 08:47:18.299,2018-10-24 00:14:33.013,1,,0,1,...,157.246757,,,,F,,201,2017-09-30 15:47:18.299000-07:00,2017-09-30 22:47:18.299000+00:00,None
2,3,20000003,2017-09-30 15:47:22.949,949,2017-09-30 08:47:22.949,2018-10-24 00:14:33.046,1,,0,1,...,128.515843,,,,F,,201,2017-09-30 15:47:22.949000-07:00,2017-09-30 22:47:22.949000+00:00,None
3,4,20000004,2017-09-30 15:47:31.418,418,2017-09-30 08:47:31.418,2018-10-24 00:14:33.112,1,,0,1,...,135.772882,,,,F,,201,2017-09-30 15:47:31.418000-07:00,2017-09-30 22:47:31.418000+00:00,None
4,5,20000005,2017-09-30 15:47:31.631,631,2017-09-30 08:47:31.631,2018-10-24 00:14:33.112,1,,0,1,...,158.083705,,,,F,,201,2017-09-30 15:47:31.631000-07:00,2017-09-30 22:47:31.631000+00:00,None



We will use the "Original" date time column. This is because this matches the wave file name. As part into the Dos Ciba database, we addressed a shortcoming in instrument programming. Specifically, the instruments do not account for daylight savings transitions.

Instead, 
Because the UTC time in pamguard is actual UTC time, the detection at it's actual time may be off by an hour or more. Using the original date-time data will allow us to be consistent with other data and metadata (i.e. the wave file names, other processed data-products).

This specific column name is what subsequent processing tools utilize.




In [3]:
pg_dataframe.drop(columns=['Local_Time', 'UTC_Time', 'Fixed Localized'], inplace=True)
pg_dataframe.rename(columns={"Original":"UTC"}, inplace=True)
pg_dataframe['Source'] = "DosDB"

 ## Obtain calls that are 1Khz or less

In [4]:
sub_1khz = pd.read_excel(r"D:\beluga\Data\Special_Data\raw\216D_PG_LF_calls.xlsx")

sub_1khz['Source'] = "Sub1Khz"
sub_1khz['Deployment'] = 216
#UTC is already a datetime64

# Get False Beluga Calls

In [6]:
bootstrap_237 = pd.read_csv(r"D:\beluga\Data\Special_Data\processed\fp_237_training_data_bootstrap.csv", parse_dates=True)

#new column casting new detection time string to datetime 64
bootstrap_237['UTC'] = pd.to_datetime(bootstrap_237["UTC"])
bootstrap_237['Source'] = "AB_ML_237_VALIDATION"
bootstrap_237['Deployment'] = 237

In [7]:
bootstrap_237.head()

,Unnamed: 0,Id,UID,Species,AB comments,UTC,Source,Deployment
0,400,116535,12590000004,F,mechanical noise,2019-09-07 00:18:11,AB_ML_237_VALIDATION,237
1,401,116536,12590000005,F,mechanical noise,2019-09-07 00:18:11,AB_ML_237_VALIDATION,237
2,402,116537,12590000006,F,mechanical noise,2019-09-07 00:18:11,AB_ML_237_VALIDATION,237
3,403,116538,12590000007,F,mechanical noise,2019-09-07 00:18:11,AB_ML_237_VALIDATION,237
4,404,116539,12590000008,F,mechanical noise,2019-09-07 00:18:11,AB_ML_237_VALIDATION,237


# Merge All Data Sources into a combined dataframe

In [8]:
combined_df = pd.DataFrame(columns=['UTC','Species','Source', 'Deployment'])
combined_df = pd.concat([combined_df, bootstrap_237, sub_1khz, pg_dataframe], join="inner")
combined_df.head()

,UTC,Species,Source,Deployment
0,2019-09-07 00:18:11,F,AB_ML_237_VALIDATION,237
1,2019-09-07 00:18:11,F,AB_ML_237_VALIDATION,237
2,2019-09-07 00:18:11,F,AB_ML_237_VALIDATION,237
3,2019-09-07 00:18:11,F,AB_ML_237_VALIDATION,237
4,2019-09-07 00:18:11,F,AB_ML_237_VALIDATION,237


In [9]:
combined_df.to_csv('updated_combined_training_dataset.csv', index=False)